In [1]:
import torch
import torch.nn as nn

torch.manual_seed(0)

# 參數
B = 1   # batch_size
K = 3   # 時間步數 (例如 3 個月)
H = 2   # 每個時間步的 hidden 維度
heads = 1

# 建一個固定的序列 x: [B, K, H]
# 這裡用手刻的數字，方便你觀察
x = torch.tensor(
    [[[1.0, 0.0],   # t1
      [0.0, 1.0],   # t2
      [1.0, 1.0]]], # t3
    dtype=torch.float32
)  # shape [1,3,2]

print("x =")
print(x)
print("x shape:", x.shape)  # [1,3,2]

# 建一個 MultiheadAttention（batch_first=True）
mha = nn.MultiheadAttention(
    embed_dim=H,
    num_heads=heads,
    batch_first=True,   # 輸入: [B, K, H]
    dropout=0.0,
)

# --------- 1. 不加 mask 的 self-attention ---------
out_free, w_free = mha(x, x, x, need_weights=True)  # w_free: [B, K, K]

print("\n=== No mask ===")
print("attn_weights (shape {}):".format(w_free.shape))
# w_free[0] 是第 1 個 batch 的 [K,K] 權重矩陣
print(w_free[0])

# 說明：w_free[0, t, s] = 在更新第 t 個時間步時，對第 s 個時間步的權重


# --------- 2. 建 causal mask：不看未來 ---------
def get_causal_mask(K, device):
    # True 表示「不能看」
    mask = torch.triu(torch.ones(K, K, dtype=torch.bool, device=device), diagonal=1)
    # True -> -inf, False -> 0.0
    attn_mask = mask.masked_fill(mask, float("-inf")).masked_fill(~mask, 0.0)
    return attn_mask

attn_mask = get_causal_mask(K, x.device)

print("\ncausal attn_mask =")
print(attn_mask)  # [K,K]

# --------- 3. 加上 causal mask 的 self-attention ---------
out_causal, w_causal = mha(x, x, x, attn_mask=attn_mask, need_weights=True)

print("\n=== With causal mask (no looking into the future) ===")
print("attn_weights (shape {}):".format(w_causal.shape))
print(w_causal[0])   # [K,K]

# 再額外印每一列，幫你對應 t=1,2,3 的含義
for t in range(K):
    print(f"\nRow t={t} (time step {t+1}) weights over s=1..{K}:")
    print(w_causal[0, t])

x =
tensor([[[1., 0.],
         [0., 1.],
         [1., 1.]]])
x shape: torch.Size([1, 3, 2])

=== No mask ===
attn_weights (shape torch.Size([1, 3, 3])):
tensor([[0.3312, 0.3263, 0.3424],
        [0.3415, 0.3720, 0.2865],
        [0.3401, 0.3650, 0.2950]], grad_fn=<SelectBackward0>)

causal attn_mask =
tensor([[False,  True,  True],
        [False, False,  True],
        [False, False, False]])

=== With causal mask (no looking into the future) ===
attn_weights (shape torch.Size([1, 3, 3])):
tensor([[1.0000, 0.0000, 0.0000],
        [0.4786, 0.5214, 0.0000],
        [0.3401, 0.3650, 0.2950]], grad_fn=<SelectBackward0>)

Row t=0 (time step 1) weights over s=1..3:
tensor([1., 0., 0.], grad_fn=<SelectBackward0>)

Row t=1 (time step 2) weights over s=1..3:
tensor([0.4786, 0.5214, 0.0000], grad_fn=<SelectBackward0>)

Row t=2 (time step 3) weights over s=1..3:
tensor([0.3401, 0.3650, 0.2950], grad_fn=<SelectBackward0>)
